In [ ]:
# xca password: xcapassword$
# xca db: betfair-api-test.xdb
# xca key: betfair-api-test-key
# betfair app name: test-app-1234562341678

In [ ]:
!pip install betfairlightweight[speed]

In [ ]:
# resources
## https://developer.betfair.com/get-started/#exchange-api
## https://betfair-developer-docs.atlassian.net/wiki/spaces/1smk3cen4v3lu3yomq5qye0ni/pages/2687673/Certificate+Generation+With+XCA
## https://betfair-developer-docs.atlassian.net/wiki/spaces/1smk3cen4v3lu3yomq5qye0ni/pages/2687915/Non-Interactive+bot+login
## https://github.com/betfair-datascientists/API/blob/master/python/API%20Tutorial.ipynb


# to enable automated access there are a few steps

## Create Betfair account

## Create a development app key: https://apps.betfair.com/visualisers/api-ng-account-operations/
### Note: Create a VERY unique name, I think app names must be unique across every single app in Betfair

## Download XCA: https://sourceforge.net/projects/xca/
### Note: You'll need to create a database to store any following objects in, don't forget the password for it!

## Create the Private Key, Cert Signing Request & Cert using: https://betfair-developer-docs.atlassian.net/wiki/spaces/1smk3cen4v3lu3yomq5qye0ni/pages/2687673/Certificate+Generation+With+XCA
### Note: Exporting the Cert as a .crt is fine, 
### Note: It;s not mentioned but the key needs to also be exported. the key will naturally export as a .pem, so manually change it to a .key

## Upload .crt to Betfair: https://myaccount.betfair.com/accountdetails/mysecurity?showAPI=1
### Note: See https://betfair-developer-docs.atlassian.net/wiki/spaces/1smk3cen4v3lu3yomq5qye0ni/pages/2687915/Non-Interactive+bot+login

## Move the .key and .crt to the certs directory that is in the same directory as your code

## Should work :)

In [1]:
# Import libraries
import betfairlightweight
from betfairlightweight import filters
import datetime as dt
import inflection
import pandas as pd
import numpy as np
import os
import datetime
import json

# Change this certs path to wherever you're storing your certificates
certs_path = os.path.join(os.path.abspath('.'), "certs")

# Change these login details to your own
my_username = "betsfrombush"
my_password = ""
my_app_key = "MAqOPVlXs3sQWpsZ"

#APP_KEY = 'MAqOPVlXs3sQWpsZ'
#SESSION = 'Pzr5zkcV7nrr9/z6SGCYewKD+3U9jYqQ7Qy2MdQ6ObY='

trading = betfairlightweight.APIClient(
    username=my_username,
    password=my_password,
    app_key=my_app_key,
    certs=certs_path
)

trading.login()

<LoginResource>

In [15]:
# Grab all event type ids. 
# This will return a list which we will iterate over to print out the id and the name of the sport
event_types = trading.betting.list_event_types()

sport_ids = pd.DataFrame({
    'sport': [event_type_object.event_type.name for event_type_object in event_types],
    'id': [event_type_object.event_type.id for event_type_object in event_types]
}).set_index('sport').sort_index()

sport_ids

,id
sport,
American Football,6423
Australian Rules,61420
Baseball,7511
Basketball,7522
Boxing,6
Cricket,4
Cycling,11
Darts,3503
Esports,27454571


In [29]:
# Filter for just football
football_filter = betfairlightweight.filters.market_filter(text_query='Soccer')

# This returns a list
football_event_type = trading.betting.list_event_types(
    filter=football_filter
)

football_event_id = football_event_type[0].event_type.id
football_event_id

'1'

In [31]:
# get next week date
next_week_date = (dt.datetime.utcnow() + dt.timedelta(weeks=1)).strftime("%Y-%m-%dT%TZ")

# Create a competition filter
competition_filter = betfairlightweight.filters.market_filter(
    event_type_ids=[football_event_id],
    market_start_time={
        'to': next_week_date
    }
)

# Get a list of competitions for soccer
competitions = trading.betting.list_competitions(
    filter=competition_filter
)

# Iterate over the competitions and create a dataframe of competitions and competition ids
football_competitions = pd.DataFrame(
    {
        'competition': [competition_object.competition.name for competition_object in competitions],
        'id': [competition_object.competition.id for competition_object in competitions]
    }
)

football_competitions.astype({'id': 'int32'}).sort_values(by='id')

,competition,id
3,Austrian Erste Liga,9
5,Dutch Eerste Divisie,11
7,Brazilian Serie A,13
8,Bulgarian A League,15
9,Croatian HNL,17
...,...,...
60,English U21 Premier League Cup,12722076
127,Czech MSFL,12736789
110,Uruguayan Cup,12751723
35,UEFA Ladies Europa Cup,12759291


In [35]:
prem_id = football_competitions[football_competitions['competition'].str.contains('English Premier League')]['id'].item()
prem_id

'10932509'

In [86]:
# Define a market filter
premier_league_event_filter = betfairlightweight.filters.market_filter(
    competition_ids=[prem_id],
    #market_countries=['UK'],
    market_start_time={
        'to': next_week_date
    }
)

# Print the filter
premier_league_event_filter

{'competitionIds': ['10932509'],
 'marketStartTime': {'to': '2025-10-22T10:27:58Z'}}

In [ ]:
# Get a list of all prem events as objects
premier_league_events = trading.betting.list_events(
    filter=premier_league_event_filter
)

premier_league_events_df = pd.DataFrame(
    [json.loads(event.json().decode('utf-8'))['event'] for event in premier_league_events]
).rename(
    columns=inflection.underscore
)

premier_league_events_df

In [238]:
premier_league_events_df

,id,name,country_code,timezone,open_date
0,34782627,Tottenham v Aston Villa,GB,GMT,2025-10-19T13:00:00.000Z
1,34782624,Sunderland v Wolves,GB,GMT,2025-10-18T14:00:00.000Z
2,34782800,Man City v Everton,GB,GMT,2025-10-18T14:00:00.000Z
3,34782625,Burnley v Leeds,GB,GMT,2025-10-18T14:00:00.000Z
4,34782727,Liverpool v Man Utd,GB,GMT,2025-10-19T15:30:00.000Z
5,34782629,Fulham v Arsenal,GB,GMT,2025-10-18T16:30:00.000Z
6,34782618,Brighton v Newcastle,GB,GMT,2025-10-18T14:00:00.000Z
7,34782635,Crystal Palace v Bournemouth,GB,GMT,2025-10-18T14:00:00.000Z
8,2022802,English Premier League,GB,Europe/London,2025-08-16T14:00:01.000Z
9,34782623,Nottm Forest v Chelsea,GB,GMT,2025-10-18T11:30:00.000Z


In [88]:
# Define market filters

event_markets = []

for event in premier_league_events_df['id'].tolist():
    market_types_filter = betfairlightweight.filters.market_filter(
        event_ids=[event]
    )

    # Request market types
    market_types = trading.betting.list_market_types(
        filter=market_types_filter
    )
    
    event_markets_df = pd.DataFrame(
        [json.loads(event.json().decode('utf-8')) for event in market_types]
    )
    
    event_markets_df['event_id'] = event

    event_markets.append(event_markets_df)

premier_league_event_markets = pd.concat(event_markets).reset_index(drop=True).rename(columns=inflection.underscore)
premier_league_event_markets

,market_type,market_count,event_id
0,ALT_TOTAL_GOALS,1,34782627
1,CORRECT_SCORE,1,34782627
2,OVER_UNDER_25,1,34782627
3,MATCH_ODDS,1,34782627
4,BOTH_TEAMS_TO_SCORE,1,34782627
...,...,...,...
76,BOTH_TEAMS_TO_SCORE,1,34782621
77,ALT_TOTAL_GOALS,1,34782621
78,OVER_UNDER_25,1,34782621
79,CORRECT_SCORE,1,34782621


In [296]:
event_catalogues = []

for event in premier_league_events_df['id'].tolist():
    market_catalogue_filter = betfairlightweight.filters.market_filter(
        event_ids=[event]
    )

    # Request market catalogues
    market_catalogues = trading.betting.list_market_catalogue(
        filter=market_catalogue_filter,
        market_projection=['RUNNER_DESCRIPTION'],
        max_results='100'
    )
    
    event_catalogues_df = pd.DataFrame(
        [json.loads(event.json().decode('utf-8')) for event in market_catalogues]
    )
    
    event_catalogues_df['event_id'] = event
    
    event_catalogues.append(event_catalogues_df)
    
premier_league_event_catalogues = pd.concat(event_catalogues).rename(columns=inflection.underscore)
premier_league_event_catalogues = premier_league_event_catalogues.explode(column='runners')
premier_league_event_catalogues[['selection_id', 'selection_name', 'handicap']] = premier_league_event_catalogues['runners'].apply(lambda x: pd.Series([x['selectionId'], x['runnerName'], x['handicap']]))
premier_league_event_catalogues = premier_league_event_catalogues.drop(columns='runners').reset_index(drop=True)
premier_league_event_catalogues

,market_id,market_name,total_matched,event_id,selection_id,selection_name,handicap
0,1.248325592,Goal Lines,1571.21,34782627,7044483,Under,0.50
1,1.248325592,Goal Lines,1571.21,34782627,7044482,Over,0.50
2,1.248325592,Goal Lines,1571.21,34782627,7044483,Under,0.75
3,1.248325592,Goal Lines,1571.21,34782627,7044482,Over,0.75
4,1.248325592,Goal Lines,1571.21,34782627,7044483,Under,1.00
...,...,...,...,...,...,...,...
2102,1.248327118,Asian Handicap,2222.65,34782621,78864,Brentford,-3.50
2103,1.248327118,Asian Handicap,2222.65,34782621,48756,West Ham,3.75
2104,1.248327118,Asian Handicap,2222.65,34782621,78864,Brentford,-3.75
2105,1.248327118,Asian Handicap,2222.65,34782621,48756,West Ham,4.00


In [297]:
premier_league_event_catalogues.query('event_id == "2022802"')

,market_id,market_name,total_matched,event_id,selection_id,selection_name,handicap
1440,1.244214540,Winner,3357952.72,2022802,1096,Arsenal,0.0
1441,1.244214540,Winner,3357952.72,2022802,56323,Liverpool,0.0
1442,1.244214540,Winner,3357952.72,2022802,47999,Man City,0.0
1443,1.244214540,Winner,3357952.72,2022802,55190,Chelsea,0.0
1444,1.244214540,Winner,3357952.72,2022802,48224,Tottenham,0.0
...,...,...,...,...,...,...,...
1750,1.247127969,PFA Player of the Year 2025/26,663.74,2022802,7649532,Youri Tielemans,0.0
1751,1.247127969,PFA Player of the Year 2025/26,663.74,2022802,36720898,Rasmus Hojlund,0.0
1752,1.247127969,PFA Player of the Year 2025/26,663.74,2022802,8767691,James Maddison,0.0
1753,1.247127969,PFA Player of the Year 2025/26,663.74,2022802,36441120,Nicolas Jackson,0.0


In [298]:
premier_league_event_catalogues['market_name'].unique()

array(['Goal Lines', 'Correct Score', 'Over/Under 2.5 Goals',
       'Match Odds', 'Both teams to Score?', 'Over/Under 0.5 Goals',
       'Asian Handicap', 'Over/Under 6.5 Goals', 'Half Time Score',
       'Over/Under 5.5 Goals', 'Over/Under 4.5 Goals', 'Half Time',
       'Over/Under 1.5 Goals', 'Half Time/Full Time',
       'Over/Under 3.5 Goals', 'Winner', 'Top 2 Finish', 'Top 4 Finish',
       'Top 6 Finish', 'Top 10 Finish', 'Relegation', 'To Finish Bottom',
       'Top Goalscorer', 'Handicap Win 2025/26',
       'Next Manager to Leave - Please Read Rules',
       'PFA Player of the Year 2025/26'], dtype=object)

In [123]:
market_book_runners = []
match_markets = [
    'Goal Lines', 
    'Correct Score', 
    'Over/Under 2.5 Goals',
    'Match Odds', 
    'Both teams to Score?', 
    'Over/Under 0.5 Goals',
    'Asian Handicap'
]

market_book_runners = [{market: []} for market in match_markets]
market_book_runners

[{'Goal Lines': []},
 {'Correct Score': []},
 {'Over/Under 2.5 Goals': []},
 {'Match Odds': []},
 {'Both teams to Score?': []},
 {'Over/Under 0.5 Goals': []},
 {'Asian Handicap': []}]

In [233]:
#%%time
# Request market books
import copy
from tqdm import tqdm

market_book_metadata = []

match_markets = [
    'Goal Lines', 
    'Correct Score', 
    'Over/Under 2.5 Goals',
    'Match Odds', 
    'Both teams to Score?', 
    'Over/Under 0.5 Goals',
    'Asian Handicap'
]

market_book_runners = {
    market: [] if market not in ['Goal Lines', 'Asian Handicap'] else {'runners': [], 'key_line': []}
    for market in match_markets
}

# Create a price filter. 
# Get all traded and offer data
price_filter = betfairlightweight.filters.price_projection(
    price_data=['EX_BEST_OFFERS']
)

for ind, market in premier_league_event_catalogues.iterrows():
    
    # we also pull league markets like winner, top scorer etc., just skip for now
    # looking at the structure it should be fairly similar
    if market['market_name'] in match_markets:
        
        market_books = trading.betting.list_market_book(
            market_ids=[market['market_id']],
            price_projection=price_filter
        )
        
        for book in market_books:
        
            book_json = json.loads(book.json().decode('utf-8'))

            book_metadata = {k: v for k, v in book_json.items() if k not in ['runners', 'keyLineDescription']}
            book_metadata['market_id'] = market['market_id']
            
            book_line = book_json['keyLineDescription']['keyLine'] if 'keyLineDescription' in book_json.keys() else None
            book_runners = book_json['runners']
            
            # pull out best prices and sizes
            exchange_prices_sizes = [runner['ex'] for runner in book_runners]
            back_exchange_prices_sizes = [
                {
                    f'back_{k}': v for k, v in runner['availableToBack'][0].items()
                } if len(runner['availableToBack']) > 0 else
                {
                    'back_price': np.nan, 'back_size': np.nan
                }
                for runner in exchange_prices_sizes
            ] # extract the best one
            lay_exchange_prices_sizes = [
                {
                    f'lay_{k}': v for k, v in runner['availableToLay'][0].items()
                } if len(runner['availableToLay']) > 0 else
                {
                    'lay_price': np.nan, 'lay_size': np.nan
                }
                for runner in exchange_prices_sizes
            ] # extract the best one
            
            prices_sizes = [
                {
                    **back, 
                    **lay
                } for back, lay in zip(
                    back_exchange_prices_sizes, 
                    lay_exchange_prices_sizes
                )
            ]
            
            book_runners = [
                {
                    **{k: v for k, v in runner.items() if k != 'ex'},
                    **price_size
                } for runner, price_size in zip(
                    book_runners,
                    prices_sizes
                )
            ]
            book_runners = [dict(d, market_id=market['market_id']) for d in book_runners]

            market_book_metadata.append(book_metadata)

            if book_line is not None:
                book_line = [dict(d, market_id=market['market_id']) for d in book_line]
                market_book_runners[market['market_name']]['runners'].extend(copy.deepcopy(book_runners))
                market_book_runners[market['market_name']]['key_line'].extend(copy.deepcopy(book_line))
            else:
                market_book_runners[market['market_name']].extend(copy.deepcopy(book_runners))

In [251]:
dir(book.runners[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_item_name_to_attribute_name_overrides',
 'adjustment_factor',
 'bet_delay_models',
 'ex',
 'get',
 'handicap',
 'last_price_traded',
 'matches',
 'matches_by_strategy',
 'orders',
 'removal_date',
 'selection_id',
 'sp',
 'status',
 'suspend_reason',
 'total_matched']

In [236]:
pd.DataFrame(market_book_runners['Match Odds'])

,selectionId,handicap,status,lastPriceTraded,totalMatched,back_price,back_size,lay_price,lay_size,market_id
0,48224,0.0,ACTIVE,2.20,0.0,2.20,38.03,2.22,53.97,1.248325567
1,63908,0.0,ACTIVE,3.60,0.0,3.50,24.06,3.60,42.34,1.248325567
2,58805,0.0,ACTIVE,3.80,0.0,3.75,402.64,3.80,622.32,1.248325567
3,62783,0.0,ACTIVE,2.52,0.0,2.52,39.15,2.54,53.79,1.248326310
4,48044,0.0,ACTIVE,3.35,0.0,3.30,423.91,3.40,264.28,1.248326310
5,58805,0.0,ACTIVE,3.25,0.0,3.20,5543.97,3.25,4825.51,1.248326310
6,47999,0.0,ACTIVE,1.42,0.0,1.42,4229.87,1.43,1005.21,1.248324306
7,56343,0.0,ACTIVE,8.80,0.0,8.60,214.14,8.80,69.34,1.248324306
8,58805,0.0,ACTIVE,5.30,0.0,5.30,685.59,5.40,729.88,1.248324306
9,56299,0.0,ACTIVE,3.50,0.0,3.50,74.95,3.55,111.51,1.248325847


In [237]:
pd.DataFrame(market_book_runners['Goal Lines']['runners'])

,selectionId,handicap,status,totalMatched,back_price,back_size,lay_price,lay_size,market_id,lastPriceTraded
0,7044483,0.50,ACTIVE,0.0,13.50,250.25,17.67,1064.57,1.248325592,NaN
1,7044482,0.50,ACTIVE,0.0,1.06,17746.13,1.08,3134.03,1.248325592,1.08
2,7044483,0.75,ACTIVE,0.0,10.80,11.92,16.15,543.47,1.248325592,NaN
3,7044482,0.75,ACTIVE,0.0,1.06,18330.08,1.11,179.18,1.248325592,NaN
4,7044483,1.00,ACTIVE,0.0,9.98,13.73,14.64,271.61,1.248325592,NaN
...,...,...,...,...,...,...,...,...,...,...
655,7044482,8.00,ACTIVE,0.0,9.33,13.78,NaN,NaN,1.248327177,NaN
656,7044483,8.25,ACTIVE,0.0,NaN,NaN,1.09,14.83,1.248327177,NaN
657,7044482,8.25,ACTIVE,0.0,9.33,13.78,NaN,NaN,1.248327177,NaN
658,7044483,8.50,ACTIVE,0.0,NaN,NaN,1.09,14.83,1.248327177,NaN


In [ ]:
trading.betting.

In [301]:
# join prices to the metadata so I know what I'm looking at
test_data = premier_league_events_df.merge(
    premier_league_event_catalogues, how='left', left_on='id', right_on='event_id'
).merge(
    pd.DataFrame(market_book_runners['Match Odds']).rename(columns=inflection.underscore), how='right', on=['market_id', 'selection_id']
)

test_data

,id,name,country_code,timezone,open_date,market_id,market_name,total_matched_x,event_id,selection_id,selection_name,handicap_x,handicap_y,status,last_price_traded,total_matched_y,back_price,back_size,lay_price,lay_size
0,34782627,Tottenham v Aston Villa,GB,GMT,2025-10-19T13:00:00.000Z,1.248325567,Match Odds,9982.83,34782627,48224,Tottenham,0.0,0.0,ACTIVE,2.20,0.0,2.20,38.03,2.22,53.97
1,34782627,Tottenham v Aston Villa,GB,GMT,2025-10-19T13:00:00.000Z,1.248325567,Match Odds,9982.83,34782627,63908,Aston Villa,0.0,0.0,ACTIVE,3.60,0.0,3.50,24.06,3.60,42.34
2,34782627,Tottenham v Aston Villa,GB,GMT,2025-10-19T13:00:00.000Z,1.248325567,Match Odds,9982.83,34782627,58805,The Draw,0.0,0.0,ACTIVE,3.80,0.0,3.75,402.64,3.80,622.32
3,34782624,Sunderland v Wolves,GB,GMT,2025-10-18T14:00:00.000Z,1.248326310,Match Odds,8434.41,34782624,62783,Sunderland,0.0,0.0,ACTIVE,2.52,0.0,2.52,39.15,2.54,53.79
4,34782624,Sunderland v Wolves,GB,GMT,2025-10-18T14:00:00.000Z,1.248326310,Match Odds,8434.41,34782624,48044,Wolves,0.0,0.0,ACTIVE,3.35,0.0,3.30,423.91,3.40,264.28
5,34782624,Sunderland v Wolves,GB,GMT,2025-10-18T14:00:00.000Z,1.248326310,Match Odds,8434.41,34782624,58805,The Draw,0.0,0.0,ACTIVE,3.25,0.0,3.20,5543.97,3.25,4825.51
6,34782800,Man City v Everton,GB,GMT,2025-10-18T14:00:00.000Z,1.248324306,Match Odds,39372.95,34782800,47999,Man City,0.0,0.0,ACTIVE,1.42,0.0,1.42,4229.87,1.43,1005.21
7,34782800,Man City v Everton,GB,GMT,2025-10-18T14:00:00.000Z,1.248324306,Match Odds,39372.95,34782800,56343,Everton,0.0,0.0,ACTIVE,8.80,0.0,8.60,214.14,8.80,69.34
8,34782800,Man City v Everton,GB,GMT,2025-10-18T14:00:00.000Z,1.248324306,Match Odds,39372.95,34782800,58805,The Draw,0.0,0.0,ACTIVE,5.30,0.0,5.30,685.59,5.40,729.88
9,34782625,Burnley v Leeds,GB,GMT,2025-10-18T14:00:00.000Z,1.248325847,Match Odds,14658.07,34782625,56299,Burnley,0.0,0.0,ACTIVE,3.50,0.0,3.50,74.95,3.55,111.51


In [309]:
# place an order - apparently minimum of £1
# just do 1 game and pick the fave
# prepare the data

selected_event = '34782627'
selected_event_data = test_data.query(f'id == "{selected_event}"')

selected_bet_data = selected_event_data[selected_event_data['last_price_traded'] == selected_event_data['last_price_traded'].min()]
selected_bet_data

,id,name,country_code,timezone,open_date,market_id,market_name,total_matched_x,event_id,selection_id,selection_name,handicap_x,handicap_y,status,last_price_traded,total_matched_y,back_price,back_size,lay_price,lay_size
0,34782627,Tottenham v Aston Villa,GB,GMT,2025-10-19T13:00:00.000Z,1.248325567,Match Odds,9982.83,34782627,48224,Tottenham,0.0,0.0,ACTIVE,2.2,0.0,2.2,38.03,2.22,53.97


In [310]:
# construct the order

limit_order_filter = betfairlightweight.filters.limit_order(
    size=1,  # stake
    price=selected_bet_data['back_price'].item(),  # odds
    persistence_type='LAPSE'
)

# Define an instructions filter
instructions_filter = betfairlightweight.filters.place_instruction(
    selection_id=str(selected_bet_data['selection_id'].item()),
    order_type="LIMIT",
    side="BACK",
    limit_order=limit_order_filter
)

instructions_filter

{'orderType': 'LIMIT',
 'selectionId': '48224',
 'side': 'BACK',
 'limitOrder': {'price': 2.2, 'persistenceType': 'LAPSE', 'size': 1}}

In [311]:
# Place the order
order = trading.betting.place_orders(
    market_id=str(selected_bet_data['market_id'].item()), # The market id we obtained from before
    customer_strategy_ref='back_the_fav',
    instructions=[instructions_filter] # This must be a list
)

In [314]:
order.__dict__

{'elapsed_time': 0.2833101749420166,
 '_datetime_created': datetime.datetime(2025, 10, 15, 16, 44, 0, 395520),
 '_datetime_updated': datetime.datetime(2025, 10, 15, 16, 44, 0, 395520),
 '_data': {'status': 'SUCCESS',
  'marketId': '1.248325567',
  'instructionReports': [{'status': 'SUCCESS',
    'instruction': {'selectionId': 48224,
     'limitOrder': {'size': 1.0, 'price': 2.2, 'persistenceType': 'LAPSE'},
     'orderType': 'LIMIT',
     'side': 'BACK'},
    'betId': '405340831725',
    'placedDate': '2025-10-15T16:44:00.000Z',
    'averagePriceMatched': 2.2,
    'sizeMatched': 1.0,
    'orderStatus': 'EXECUTION_COMPLETE'}]},
 'market_id': '1.248325567',
 'status': 'SUCCESS',
 'customer_ref': None,
 'error_code': None,
 'place_instruction_reports': [<betfairlightweight.resources.bettingresources.PlaceOrderInstructionReports at 0x228bcd71820>]}

In [7]:
# check my orders for my strategy
pd.DataFrame(trading.betting.list_current_orders(customer_strategy_refs=['back_the_fav']).__dict__['_data']['currentOrders'])

,betId,marketId,selectionId,handicap,priceSize,bspLiability,side,status,persistenceType,orderType,placedDate,matchedDate,averagePriceMatched,sizeMatched,sizeRemaining,sizeLapsed,sizeCancelled,sizeVoided,regulatorCode,customerStrategyRef
0,405340831725,1.248325567,48224,0.0,"{'price': 2.2, 'size': 1.0}",0.0,BACK,EXECUTION_COMPLETE,LAPSE,LIMIT,2025-10-15T16:44:00.000Z,2025-10-15T16:44:00.000Z,2.2,1.0,0.0,0.0,0.0,0.0,GIBRALTAR REGULATOR,back_the_fav


In [8]:
#
# listClearedOrders (need the market to settle first to get anything back)
cleared_orders = trading.betting.list_cleared_orders(bet_status="SETTLED", market_ids=["1.248325567"])
pd.DataFrame(cleared_orders._data['clearedOrders'])

""


In [9]:
# market P&L
# note this only works with a market ID, which will clear soon after the event finishes
# but you can pull this immediately after placement to understand potential P&L

# Get the profit/loss - this returns a list
pl = trading.betting.list_market_profit_and_loss(
    market_ids=["1.248325567"], 
    include_bsp_bets='true', 
    include_settled_bets='true'
)

pl_df = pd.DataFrame(pl[0]._data['profitAndLosses']).assign(marketId=pl[0].market_id)
pl_df

,selectionId,ifWin,marketId
0,48224,1.2,1.248325567
1,63908,-1.0,1.248325567
2,58805,-1.0,1.248325567


In [10]:
# account statement

# Define a date filter - get all bets for the past 7 days
four_days_ago = (datetime.datetime.utcnow() - datetime.timedelta(days=7)).strftime("%Y-%m-%dT%TZ")
acct_statement_date_filter = betfairlightweight.filters.time_range(from_=four_days_ago)

# Request account statement
account_statement = trading.account.get_account_statement(item_date_range=acct_statement_date_filter)

# Create df of recent transactions
recent_transactions = pd.DataFrame(account_statement._data['accountStatement'])
recent_transactions

,refId,itemDate,amount,balance,itemClassData,legacyData,itemClass
0,0,2025-10-15T15:25:00.000Z,10.0,10.0,"{'unknownStatementItem': '{""avgPrice"":0.0,""bet...","{'avgPrice': 0.0, 'betSize': 0.0, 'betType': '...",UNKNOWN


In [12]:
# Create df of itemClassData 
# iterate over the account statement list and convert to json so that the DataFrame function
# can read it correctly
class_data = [
    json.loads(account_statement.account_statement[i].item_class_data['unknownStatementItem']) 
    for i in range(len(account_statement.account_statement))
]
class_df = pd.DataFrame(class_data)
class_df

,avgPrice,betSize,betType,betCategoryType,commissionRate,eventId,eventTypeId,fullMarketName,grossBetAmount,marketName,...,placedDate,selectionId,selectionName,handicap,startDate,transactionType,transactionId,winLose,deadHeatPriceDivisor,avgPriceRaw
0,0.0,0.0,B,E,None,0,0,Instant Bank Transfer deposit,0.0,DEPOSIT,...,2025-10-15T15:25:00.000Z,0,None,None,0001-01-01T00:00:00.000Z,ACCOUNT_CREDIT,200001055685170,RESULT_NOT_APPLICABLE,None,0.0


In [13]:
trading.logout()

<LogoutResource>